# AlphaGenome Genomic Prediction

![AlphaGenome Genomic Prediction](https://proto-bio.github.io/proto-assets/images/tool/alphagenome/hero.png)

AlphaGenome is a multi-task genomic foundation model from Google DeepMind that predicts regulatory signals, splicing, and 3D contact maps from DNA sequence at single base-pair resolution. It supports multiple context lengths (16 Kb to 1 Mb) and provides six tool functions covering interval prediction, variant-effect prediction, raw sequence prediction, variant scoring, interval scoring, and in-silico saturating mutagenesis.

This notebook demonstrates four representative functions: interval prediction, variant-effect prediction, variant scoring, and in-silico mutagenesis. The remaining two functions (`run_alphagenome_predict_sequences` and `run_alphagenome_score_intervals`) follow the same patterns and share input schemas with the functions shown here.

In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("alphagenome")
display_overview("alphagenome")
display_docs_section("alphagenome", "Background")

## Available tools

In [ ]:
display_available_tools("alphagenome")

### `run_alphagenome_predict_intervals`

Given a genomic interval (chromosome, start, end), this function runs the AlphaGenome model and returns multi-track regulatory predictions for that region. The model supports RNA-seq, ATAC, DNase, histone ChIP, transcription factor ChIP, splice site, and contact map outputs. The interval is automatically resized up to the smallest supported context length that contains it. Output tracks can be filtered by ontology term (cell line or tissue) to focus on relevant biological contexts.

In [ ]:
from proto_tools import (
    AlphaGenomeInterval,
    AlphaGenomePredictIntervalsConfig,
    AlphaGenomePredictIntervalsInput,
    run_alphagenome_predict_intervals,
)

In [ ]:
# Display input docs
display_api_reference("alphagenome", "input", "run_alphagenome_predict_intervals")

# Paper Figure 2a coordinates: chr19, 1 Mb window, HepG2 cell line
interval_inputs = AlphaGenomePredictIntervalsInput(
    intervals=AlphaGenomeInterval(
        chromosome="chr19",
        interval_start=10_587_331,
        interval_end=11_635_907,
    ),
)

In [ ]:
# Display config docs
display_api_reference("alphagenome", "config", "run_alphagenome_predict_intervals")

interval_config = AlphaGenomePredictIntervalsConfig(
    requested_outputs=["RNA_SEQ", "ATAC", "DNASE", "CHIP_HISTONE", "CHIP_TF"],
    ontology_terms=["EFO:0001187"],  # HepG2
    organism="human",
    device="cuda",  # AlphaGenome is GPU-only
)

In [ ]:
# Run the interval prediction function
interval_result = run_alphagenome_predict_intervals(interval_inputs, interval_config)

In [ ]:
display_api_reference("alphagenome", "output", "run_alphagenome_predict_intervals")

output = interval_result[0]
print(f"Requested outputs: {output.requested_outputs}")
print(f"Result keys: {list(output.result.get('predictions', {}).keys())}")

### `run_alphagenome_predict_variants`

Given a variant (chromosome, context window, variant position, reference and alternate alleles), this function runs AlphaGenome twice — once for the reference sequence and once for the alternate — and returns the full track-level predictions for both alleles. This is the raw prediction function that gives access to the complete output tensor for each allele. It reproduces Figure 3c from the AlphaGenome paper: a G-to-C variant at chr21:46,126,238 in the COL6A2 gene that causes alternative splice junction formation.

In [ ]:
from proto_tools import (
    AlphaGenomePredictVariantsConfig,
    AlphaGenomePredictVariantsInput,
    AlphaGenomeVariant,
    run_alphagenome_predict_variants,
)

In [ ]:
# Display input docs
display_api_reference("alphagenome", "input", "run_alphagenome_predict_variants")

# Paper Figure 3c coordinates: chr21:46,126,238 G>C (COL6A2), 1 Mb context, Aorta tissue
variant_inputs = AlphaGenomePredictVariantsInput(
    variants=AlphaGenomeVariant(
        chromosome="chr21",
        interval_start=45_601_950,
        interval_end=46_650_526,
        variant_position=46_126_238,
        reference_bases="G",
        alternate_bases="C",
    ),
)

In [ ]:
# Display config docs
display_api_reference("alphagenome", "config", "run_alphagenome_predict_variants")

variant_config = AlphaGenomePredictVariantsConfig(
    requested_outputs=["RNA_SEQ", "SPLICE_SITE_USAGE"],
    ontology_terms=["UBERON:0001496"],  # Aorta
    organism="human",
    device="cuda",  # AlphaGenome is GPU-only
)

In [ ]:
# Run the variant-effect prediction function
variant_result = run_alphagenome_predict_variants(variant_inputs, variant_config)

In [ ]:
display_api_reference("alphagenome", "output", "run_alphagenome_predict_variants")

output = variant_result[0]
print(f"Variant metadata: {output.variant}")
print(f"Result keys: {list(output.result.get('predictions', {}).keys())}")

### `run_alphagenome_score_variants`

This function evaluates variants using AlphaGenome's recommended variant scorers and returns a tidy table of effect-size scores with one row per scorer-track-gene combination. It is more convenient than raw variant-effect prediction when you need quantitative effect sizes rather than full track-level predictions. The same COL6A2 variant from the previous section is used here to demonstrate the scoring output format.

In [ ]:
import pandas as pd
from proto_tools import (
    AlphaGenomeScoreVariantsConfig,
    AlphaGenomeScoreVariantsInput,
    run_alphagenome_score_variants,
)

In [ ]:
# Display input docs
display_api_reference("alphagenome", "input", "run_alphagenome_score_variants")

score_variant_inputs = AlphaGenomeScoreVariantsInput(
    variants=AlphaGenomeVariant(
        chromosome="chr21",
        interval_start=45_601_950,
        interval_end=46_650_526,
        variant_position=46_126_238,
        reference_bases="G",
        alternate_bases="C",
    ),
)

In [ ]:
# Display config docs
display_api_reference("alphagenome", "config", "run_alphagenome_score_variants")

score_variant_config = AlphaGenomeScoreVariantsConfig(
    variant_scorers=["RNA_SEQ", "SPLICE_SITE_USAGE"],
    organism="human",
    device="cuda",  # AlphaGenome is GPU-only
)

In [ ]:
# Run the variant scoring function
score_variant_result = run_alphagenome_score_variants(score_variant_inputs, score_variant_config)

In [ ]:
display_api_reference("alphagenome", "output", "run_alphagenome_score_variants")

output = score_variant_result[0]
print(f"Returned {len(output.scores)} score records")
if output.scores:
    df = pd.DataFrame(output.scores)
    print(df.head(10).to_string())

### `run_alphagenome_score_ism_variants_batch`

In-silico saturating mutagenesis (ISM) systematically mutates every position within a sub-interval to every possible alternate base and scores each substitution. The result is a comprehensive map of positional sensitivity that reveals which bases within a regulatory element are critical for its function. This example saturate-mutates a 10 bp window around the COL6A2 variant, scoring each single-nucleotide substitution with the splice sites scorer.

In [ ]:
from proto_tools import (
    AlphaGenomeISM,
    AlphaGenomeScoreISMConfig,
    AlphaGenomeScoreISMInput,
    run_alphagenome_score_ism_variants_batch,
)

In [ ]:
# Display input docs
display_api_reference("alphagenome", "input", "run_alphagenome_score_ism_variants_batch")

ism_inputs = AlphaGenomeScoreISMInput(
    requests=AlphaGenomeISM(
        chromosome="chr21",
        interval_start=45_601_950,
        interval_end=46_650_526,
        ism_interval_start=46_126_233,
        ism_interval_end=46_126_243,  # 10 bp window around variant
    ),
)

In [ ]:
# Display config docs
display_api_reference("alphagenome", "config", "run_alphagenome_score_ism_variants_batch")

ism_config = AlphaGenomeScoreISMConfig(
    variant_scorers=["SPLICE_SITE_USAGE"],
    organism="human",
    device="cuda",  # AlphaGenome is GPU-only
)

In [ ]:
# Run the ISM scoring function
ism_result = run_alphagenome_score_ism_variants_batch(ism_inputs, ism_config)

In [ ]:
display_api_reference("alphagenome", "output", "run_alphagenome_score_ism_variants_batch")

output = ism_result[0]
print(f"Returned {len(output.scores)} score records")
if output.scores:
    df = pd.DataFrame(output.scores)
    print(df.head(10).to_string())

## Export Results

Prediction and scoring outputs can be saved to standard file formats for downstream analysis. Prediction outputs support JSON and NumPy formats, while scoring outputs support JSON and CSV. The prediction exports are very large, so they are written locally and not committed to the repository.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

interval_result[0].export("alphagenome_predict_intervals", export_path=output_dir, file_format="json")
variant_result[0].export("alphagenome_predict_variants", export_path=output_dir, file_format="json")
score_variant_result[0].export("alphagenome_score_variants", export_path=output_dir, file_format="csv")
ism_result[0].export("alphagenome_score_ism_variants_batch", export_path=output_dir, file_format="csv")